# Dataset Tuner (Eval Style)
Interactive sliders for `3D3F2C` and `ND-KF-MLP`.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()), None)

if repo_root is None:
    raise RuntimeError("Could not find project root (missing pyproject.toml in parent dirs).")

repo_root_str = str(repo_root)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

print(f"Using repo root: {repo_root}")

Using repo root: /gpfs/home3/rkabai/github/eb_jepa_private


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
from multimodal_experiments.ssl_dual_alignment.dataset import DualDisentangleDataset
import yaml

# --- 1. UI Definition ---
out_plot = widgets.Output()
out_log = widgets.Output()

saved_configs = []

style = {'description_width': 'initial'}

w_data_type = widgets.Dropdown(options=['3d-3f-2c', 'nd-kf-mlp', '2d', '3d-av-1f-common', '3d-2f-common'], value='3d-3f-2c', description='Data Type:', style=style)
w_num_samples = widgets.IntSlider(min=100, max=10000, step=100, value=2000, description='Samples:', style=style)

w_man_noise_a = widgets.FloatSlider(min=0.0, max=0.5, step=0.01, value=0.02, description='Manifold Noise A:', style=style)
w_man_noise_b = widgets.FloatSlider(min=0.0, max=0.5, step=0.01, value=0.02, description='Manifold Noise B:', style=style)

w_asym_corr_a = widgets.FloatSlider(min=0.0, max=0.5, step=0.01, value=0.0, description='Asym Corrupt A:', style=style)
w_asym_corr_b = widgets.FloatSlider(min=0.0, max=0.5, step=0.01, value=0.0, description='Asym Corrupt B:', style=style)

w_asym_mism_a = widgets.FloatSlider(min=0.0, max=0.5, step=0.01, value=0.0, description='Asym Mismatch A:', style=style)
w_asym_mism_b = widgets.FloatSlider(min=0.0, max=0.5, step=0.01, value=0.0, description='Asym Mismatch B:', style=style)

w_ext_noise = widgets.FloatSlider(min=0.0, max=0.8, step=0.05, value=0.0, description='External Noise:', style=style)
w_bbox_exp = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.25, description='BBox Expansion:', style=style)

w_k_shared = widgets.IntSlider(min=1, max=10, step=1, value=2, description='K Shared (MLP):', style=style)
w_m_unique = widgets.IntSlider(min=1, max=10, step=1, value=2, description='M Unique (MLP):', style=style)
w_d_out = widgets.IntSlider(min=3, max=64, step=1, value=16, description='D Out (MLP):', style=style)

btn_gen = widgets.Button(description="Generate & Plot", button_style='primary')
btn_save = widgets.Button(description="Save Current Config", button_style='success')
btn_export = widgets.Button(description="Export Saved to YAML", button_style='warning')

ui_left = widgets.VBox([w_data_type, w_num_samples, w_man_noise_a, w_man_noise_b, w_asym_corr_a, w_asym_corr_b])
ui_mid = widgets.VBox([w_asym_mism_a, w_asym_mism_b, w_ext_noise, w_bbox_exp])
ui_right = widgets.VBox([w_k_shared, w_m_unique, w_d_out])
ui_buttons = widgets.HBox([btn_gen, btn_save, btn_export])

ui_top = widgets.HBox([ui_left, ui_mid, ui_right])
ui_full = widgets.VBox([ui_top, ui_buttons, out_log, out_plot])

# --- 2. Plotting Logic (Eval.py Style) ---
def _get_point_type_colors(param_values, point_types):
    # Replicating eval.py logic: clean = rainbow by param, outliers = gray/black
    import matplotlib.pyplot as plt
    u_vals = param_values[:, 0] if param_values.ndim == 2 else param_values
    cmap = plt.get_cmap("rainbow")
    base_colors = [f"rgb({int(c[0]*255)},{int(c[1]*255)},{int(c[2]*255)})" for c in cmap(u_vals)]
    
    colors = []
    for i, pt in enumerate(point_types):
        pt = int(pt)
        if pt == 5: # External
            colors.append("rgb(0,0,0)") # Black
        elif pt in (2, 4): # Corrupted/Mismatched side
            colors.append("rgb(128,128,128)") # Gray
        else: # Clean
            colors.append(base_colors[i])
    return colors

def plot_dataset(ds):
    da = ds.data_a.numpy()
    db = ds.data_b.numpy()
    
    # Take first 3 dims for visualization
    if da.shape[1] > 3:
        da = da[:, :3]; db = db[:, :3]
        
    c_a = _get_point_type_colors(ds.param_values.numpy(), ds.point_type_a.numpy())
    c_b = _get_point_type_colors(ds.param_values.numpy(), ds.point_type_b.numpy())
    
    fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
                        subplot_titles=('Input Space A', 'Input Space B'))
    
    fig.add_trace(go.Scatter3d(x=da[:, 0], y=da[:, 1], z=da[:, 2],
                               mode='markers', marker=dict(size=3, color=c_a),
                               name='Mod A'), row=1, col=1)
                               
    fig.add_trace(go.Scatter3d(x=db[:, 0], y=db[:, 1], z=db[:, 2],
                               mode='markers', marker=dict(size=3, color=c_b),
                               name='Mod B'), row=1, col=2)
                               
    scene_cube = dict(aspectmode="cube")
    fig.update_layout(height=500, width=1000, showlegend=False,
                      scene=scene_cube, scene2=scene_cube,
                      margin=dict(l=0, r=0, t=50, b=0))
    return fig

# --- 3. Callbacks ---
def get_current_params():
    return {
        'data_type': w_data_type.value,
        'num_samples': w_num_samples.value,
        'manifold_noise_a': w_man_noise_a.value,
        'manifold_noise_b': w_man_noise_b.value,
        'asym_corrupt_rate_a': w_asym_corr_a.value,
        'asym_corrupt_rate_b': w_asym_corr_b.value,
        'asym_mismatch_rate_a': w_asym_mism_a.value,
        'asym_mismatch_rate_b': w_asym_mism_b.value,
        'external_noise_ratio': w_ext_noise.value,
        'noise_bbox_expansion': w_bbox_exp.value,
        'k_shared': w_k_shared.value,
        'm_unique': w_m_unique.value,
        'd_out': w_d_out.value,
        'seed': 42
    }


from IPython.display import display

def on_generate(b):
    with out_plot:
        clear_output(wait=True)
        params = get_current_params()
        with out_log:
            clear_output()
            print("Generating...")
        try:
            ds = DualDisentangleDataset(**params)
            fig = plot_dataset(ds)
            display(fig)  # use display instead of fig.show
            with out_log:
                clear_output()
                print(f"Plotted {len(ds)} samples.")
        except Exception as e:
            with out_log:
                clear_output()
                print(f"Error: {e}")

def on_save(b):
    with out_log:
        params = get_current_params(); saved_configs.append(params)
        print(f"Saved config {len(saved_configs)}")

def on_export(b):
    if not saved_configs: return
    with open("saved_tuning_configs.yaml", 'w') as f:
        yaml.dump({'saved_setups': saved_configs}, f, sort_keys=False)
    with out_log: print(f"Exported to saved_tuning_configs.yaml")

# Remove old handlers first (important when rerunning cell)
btn_gen.on_click(on_generate, remove=True)
btn_save.on_click(on_save, remove=True)
btn_export.on_click(on_export, remove=True)

# Bind once
btn_gen.on_click(on_generate)
btn_save.on_click(on_save)
btn_export.on_click(on_export)

display(ui_full)